# RAG integrity benchmark

One PDF and one question file are tested across **18** retrieval setups (3 chunkers × 3 embedders × 2 vector stores).

| Component | Options |
|-----------|----------|
| Chunkers | `fixed`, `recursive`, `semantic` |
| Embedders | `minilm`, `bge`, `e5` |
| Vector stores | `faiss`, `chroma` |

Read **How the pipeline works** next, then the metric glossary.


## How the pipeline works

The goal is to compare retrieval setups on the same document. Questions and PDF stay fixed; only chunking, the embedding model, and FAISS vs Chroma change between runs.

**Inputs**
- A PDF (for example the university handbook).
- `handbook_questions.json`: each item has a `question` and an `evidence_text` string copied from the PDF. That string is the ground truth for retrieval.

**Pipeline (same idea for all 18 runs)**

1. `pypdf` reads the PDF into one text string.
2. A chunker splits that text into many chunks (fixed, recursive, or semantic).
3. We score chunk boundaries: SBI, MSCR, and whether each question's `evidence_text` fits in a single chunk (ESI).
4. For one config (chunker + embedder + FAISS or Chroma): embed every chunk, build the index.
5. For each question: retrieve the top `k` chunks by vector similarity.
6. If `evidence_text` is found inside those chunks (whitespace normalized), count a hit. The hit rate over all questions is `evidence_hit_rate`.
7. Build a short extractive answer from the retrieved text (FLAN-T5 only if `USE_EXTRACTIVE_ONLY` is False).
8. Append one row to `results_table` and to `results.csv`.
9. Repeat step 4-8 for all 18 configs.

**What you get at the end**
- `results_table` and `results.csv`: one row per config with integrity scores, `evidence_hit_rate`, timing, and local QA proxy columns.
- Summary tables (section 14) and plots (section 15) comparing chunkers, embedders, and stores.

**What we do not do here**
- No paid API judge for RAGAS; QA columns use `local_overlap` (token overlap + MiniLM similarity).
- Answers are mainly for logging; the main mark for this project is whether the **right span** was retrieved.

Run cells from section 1 downward. Section 13 runs the full loop; it takes the longest on a large PDF.


### Metric glossary (columns in tables and CSV)

| Column | Meaning |
|--------|--------|
| `sbi_pct` | **SBI** (Sentence Boundary Integrity): share of chunk start/end edges that align with an NLTK sentence boundary (0-100, higher is cleaner). |
| `mscr_pct` | **MSCR** (Mid-Sentence Cut Rate): `100 - sbi_pct`; share of edges that cut inside a sentence. |
| `mid_start_pct` | Percent of aligned chunks whose **start** is a mid-sentence cut. |
| `mid_end_pct` | Percent of aligned chunks whose **end** is a mid-sentence cut. |
| `esi_pct` | **ESI** (Evidence Span Integrity): percent of questions whose full `evidence_text` fits inside **one** chunk of the indexed set (before retrieval). |
| `esi_n` | Number of questions used to compute `esi_pct`. |
| `evidence_hit_rate` | Primary retrieval score: fraction of questions where `evidence_text` (after whitespace collapse) appears in the **top-k** retrieved chunks. |
| `retrieved_esi_rate` | Fraction of questions where `evidence_text` fits entirely inside **one** of the top-k retrieved chunks (retrieval-time span check). |
| `num_chunks` | Chunk count for that chunker on this PDF. |
| `index_time_s` | Seconds to embed chunks and build the vector index. |
| `avg_query_time_s` | Mean seconds per question for similarity search. |
| `n_questions` | Questions evaluated in that config run. |
| `qa_metric_source` | Always `local_overlap` here: token overlap + MiniLM cosine proxies (not official RAGAS). |
| `context_precision` | Local proxy: share of top-k chunks that overlap gold/evidence tokens. |
| `context_recall` | Local proxy: recall of gold/evidence tokens in retrieved text. |
| `faithfulness` | Local proxy: share of answer tokens supported by retrieved text. |
| `answer_relevancy` | Local proxy: MiniLM cosine between question and answer (clipped to 0-1). |


## 1. Install dependencies


In [ ]:
import os

os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

!pip install -q "langchain>=0.2.16,<1.0" "langchain-core>=0.2.38,<1.0" "langchain-community>=0.2.16,<1.0"
!pip install -q "langchain-text-splitters>=0.2.0,<1.0" "langchain-experimental>=0.0.61,<0.4" "langchain-huggingface>=0.0.3,<0.2"
!pip install -q sentence-transformers faiss-cpu chromadb pypdf transformers torch
!pip install -q accelerate pandas tiktoken nltk scipy matplotlib seaborn

import nltk

nltk.download("punkt")
try:
    nltk.download("punkt_tab")
except Exception:
    pass

print("Install finished.")


## 2. Settings


In [ ]:
USE_DRIVE = False
DRIVE_PDF_PATH = "/content/drive/MyDrive/handbook/riverside_university_handbook.pdf"
DRIVE_QUESTIONS_PATH = "/content/drive/MyDrive/handbook/handbook_questions.json"

PDF_PATH = None
QUESTIONS_PATH = None

SMOKE_TEST = False
USE_EXTRACTIVE_ONLY = True
RESULTS_CSV = "/content/results.csv"

# SMOKE_TEST=True runs 2 configs and 2 questions only.


## 3. Load PDF and questions


In [ ]:
from google.colab import files

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PDF_PATH = DRIVE_PDF_PATH
    QUESTIONS_PATH = DRIVE_QUESTIONS_PATH

if (PDF_PATH is None) or (not os.path.exists(str(PDF_PATH))):
    print("Upload the handbook PDF and handbook_questions.json.")
    uploaded_files = files.upload()
    uploaded_names = list(uploaded_files.keys())
    pdf_names = [name for name in uploaded_names if name.lower().endswith(".pdf")]
    json_names = [name for name in uploaded_names if name.lower().endswith(".json")]
    if len(pdf_names) == 0:
        raise FileNotFoundError("No PDF uploaded.")
    PDF_PATH = "/content/" + pdf_names[0]
    if len(json_names) > 0:
        QUESTIONS_PATH = "/content/" + json_names[0]

print("PDF_PATH       =", PDF_PATH)
print("QUESTIONS_PATH =", QUESTIONS_PATH)
print("SMOKE_TEST     =", SMOKE_TEST)


## 4. Experiment grid


In [ ]:
CHUNKER_NAMES = ["fixed", "recursive", "semantic"]

EMBEDDING_MODELS = {
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
    "bge": "BAAI/bge-small-en-v1.5",
    "e5": "intfloat/e5-small-v2",
}

VECTOR_DATABASES = ["faiss", "chroma"]

FIXED_CHUNK_SIZE = 500
FIXED_CHUNK_OVERLAP = 50
RECURSIVE_CHUNK_SIZE = 1000
RECURSIVE_CHUNK_OVERLAP = 200
SEMANTIC_SPLIT_MODEL = EMBEDDING_MODELS["minilm"]

TOP_K = 3
MAX_CONTEXT_CHARS = 800
GENERATOR_MODEL = "google/flan-t5-small"

ALL_CONFIGS = []
for chunker_name in CHUNKER_NAMES:
    for embedding_name in EMBEDDING_MODELS:
        for vector_db_name in VECTOR_DATABASES:
            ALL_CONFIGS.append(
                {"chunker": chunker_name, "embedder": embedding_name, "vector_db": vector_db_name}
            )

assert len(ALL_CONFIGS) == 18

if SMOKE_TEST:
    CONFIGS_TO_RUN = [
        {"chunker": "fixed", "embedder": "minilm", "vector_db": "faiss"},
        {"chunker": "semantic", "embedder": "minilm", "vector_db": "faiss"},
    ]
else:
    CONFIGS_TO_RUN = ALL_CONFIGS

print("Configs to run:", len(CONFIGS_TO_RUN))
for index, config in enumerate(CONFIGS_TO_RUN, start=1):
    print(
        f"  {index:2d}. {config['chunker']:10s}  {config['embedder']:7s}  {config['vector_db']}"
    )


## 5. Extract PDF text


In [ ]:
from typing import Dict, List, Optional, Tuple
from pypdf import PdfReader


def load_pdf(pdf_path: str) -> Dict:
    reader = PdfReader(pdf_path)
    pages = []
    for page_index, page in enumerate(reader.pages):
        extracted = page.extract_text() or ""
        pages.append({"page_num": page_index + 1, "text": extracted.strip()})
    full_text = "\n\n".join(page["text"] for page in pages if page["text"])
    return {"pages": pages, "full_text": full_text, "num_pages": len(pages)}


document = load_pdf(PDF_PATH)
full_text = document["full_text"]

if len(full_text) < 200:
    raise ValueError("Too little text extracted. The PDF may be image-only.")

print("Pages:", document["num_pages"])
print("Characters:", f"{len(full_text):,}")
print(full_text[:300].replace("\n", " "))


## 6. Sentence boundaries (for SBI / MSCR)


In [ ]:
import re
import nltk

BOUNDARY_SLACK = 8


def collapse_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def find_sentence_spans(text: str) -> List[Tuple[int, int]]:
    spans = []
    search_from = 0
    for sentence in nltk.sent_tokenize(text):
        found_at = text.find(sentence, search_from)
        if found_at < 0:
            continue
        spans.append((found_at, found_at + len(sentence)))
        search_from = found_at + len(sentence)
    return spans


def is_near(value: int, candidates: List[int], slack: int = BOUNDARY_SLACK) -> bool:
    for candidate in candidates:
        if abs(value - candidate) <= slack:
            return True
    return False


sentence_spans = find_sentence_spans(full_text)
sentence_start_indexes = [start for start, end in sentence_spans]
sentence_end_indexes = [end for start, end in sentence_spans]
print("NLTK sentences:", len(sentence_spans))


## 7. Chunking


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, TokenTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings


def chunk_fixed(text: str) -> List[str]:
    splitter = TokenTextSplitter(chunk_size=FIXED_CHUNK_SIZE, chunk_overlap=FIXED_CHUNK_OVERLAP)
    return [piece.strip() for piece in splitter.split_text(text) if piece.strip()]


def chunk_recursive(text: str) -> List[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=RECURSIVE_CHUNK_SIZE,
        chunk_overlap=RECURSIVE_CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""],
        keep_separator=True,
    )
    return [piece.strip() for piece in splitter.split_text(text) if piece.strip()]


def chunk_semantic(text: str) -> List[str]:
    breakpoint_embeddings = HuggingFaceEmbeddings(model_name=SEMANTIC_SPLIT_MODEL)
    splitter = SemanticChunker(
        breakpoint_embeddings,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=90,
    )
    return [piece.strip() for piece in splitter.split_text(text) if piece.strip()]


chunk_functions = {"fixed": chunk_fixed, "recursive": chunk_recursive, "semantic": chunk_semantic}

chunk_cache: Dict[str, List[str]] = {}
needed_chunkers = sorted({config["chunker"] for config in CONFIGS_TO_RUN})

for chunker_name in needed_chunkers:
    print("Chunker:", chunker_name)
    chunk_cache[chunker_name] = chunk_functions[chunker_name](full_text)
    print("  chunks:", len(chunk_cache[chunker_name]))


## 8. SBI, MSCR, and ESI

Computed **per chunker** (same values for every embedder/DB using that chunker).

- **SBI (`sbi_pct`)**: for each chunk edge (start and end), check NLTK sentence boundaries within `BOUNDARY_SLACK` characters.
- **MSCR (`mscr_pct`)**: complement of SBI on those edges.
- **ESI (`esi_pct`)**: after questions load, checks whether each gold `evidence_text` is a substring of a single chunk (whitespace collapsed).


In [ ]:
import pandas as pd
from IPython.display import display


def inspect_chunk_boundaries(document_text: str, chunk: str) -> Dict:
    start_index = document_text.find(chunk)
    if start_index < 0:
        return {"aligned": False, "start_is_cut": None, "end_is_cut": None}
    end_index = start_index + len(chunk)
    starts_on_sentence = is_near(start_index, sentence_start_indexes) or start_index <= BOUNDARY_SLACK
    ends_on_sentence = is_near(end_index, sentence_end_indexes) or abs(end_index - len(document_text)) <= BOUNDARY_SLACK
    return {
        "aligned": True,
        "start_is_cut": not starts_on_sentence,
        "end_is_cut": not ends_on_sentence,
    }


def compute_sbi_and_mscr(chunks: List[str], chunker_name: str) -> Dict:
    number_of_edges = 0
    number_of_cuts = 0
    start_cuts = 0
    end_cuts = 0
    aligned_chunks = 0
    for chunk in chunks:
        flags = inspect_chunk_boundaries(full_text, chunk)
        if not flags["aligned"]:
            continue
        aligned_chunks += 1
        number_of_edges += 2
        if flags["start_is_cut"]:
            number_of_cuts += 1
            start_cuts += 1
        if flags["end_is_cut"]:
            number_of_cuts += 1
            end_cuts += 1
    sbi = 0.0 if number_of_edges == 0 else 100.0 * (number_of_edges - number_of_cuts) / number_of_edges
    mscr = 100.0 - sbi
    denom = max(aligned_chunks, 1)
    return {
        "chunker": chunker_name,
        "num_chunks": len(chunks),
        "aligned_chunks": aligned_chunks,
        "sbi_pct": round(sbi, 2),
        "mscr_pct": round(mscr, 2),
        "mid_start_pct": round(100.0 * start_cuts / denom, 2),
        "mid_end_pct": round(100.0 * end_cuts / denom, 2),
    }


def gold_evidence_fits_in_one_chunk(evidence_text: str, chunks: List[str]) -> bool:
    evidence = collapse_whitespace(evidence_text)
    if not evidence:
        return False
    for chunk in chunks:
        if evidence in collapse_whitespace(chunk):
            return True
    return False


def compute_esi(chunks: List[str], question_list: List[Dict]) -> Dict:
    usable = [item for item in question_list if collapse_whitespace(str(item.get("evidence_text", "")))]
    if not usable:
        return {"esi_pct": None, "esi_n": 0}
    contained = sum(
        1 for item in usable if gold_evidence_fits_in_one_chunk(item["evidence_text"], chunks)
    )
    return {"esi_pct": round(100.0 * contained / len(usable), 2), "esi_n": len(usable)}


integrity_by_chunker = {}
for chunker_name, chunks in chunk_cache.items():
    integrity_by_chunker[chunker_name] = compute_sbi_and_mscr(chunks, chunker_name)

display(pd.DataFrame(list(integrity_by_chunker.values())).style.hide(axis="index"))


## 9. Vector indexes and retrieval


In [ ]:
from langchain_community.vectorstores import FAISS, Chroma
from langchain_core.documents import Document

embedding_model_cache = {}


def get_embedding_model(embedder_key: str):
    if embedder_key not in embedding_model_cache:
        model_id = EMBEDDING_MODELS[embedder_key]
        print("  Loading embedder:", embedder_key, "->", model_id)
        embedding_model_cache[embedder_key] = HuggingFaceEmbeddings(model_name=model_id)
    return embedding_model_cache[embedder_key]


def add_e5_prefix(text: str, embedder_key: str, is_query: bool) -> str:
    if embedder_key != "e5":
        return text
    return ("query: " if is_query else "passage: ") + text


def build_index(chunks: List[str], embedder_key: str, vector_db_name: str, collection_name: str):
    embedding_model = get_embedding_model(embedder_key)
    documents = [
        Document(page_content=add_e5_prefix(chunk, embedder_key, is_query=False), metadata={"i": i, "raw": chunk})
        for i, chunk in enumerate(chunks)
    ]
    if vector_db_name == "faiss":
        return FAISS.from_documents(documents, embedding=embedding_model)
    if vector_db_name == "chroma":
        return Chroma.from_documents(documents, embedding=embedding_model, collection_name=collection_name[:63])
    raise ValueError(vector_db_name)


def retrieve_chunks(index, question: str, embedder_key: str, k: int = TOP_K) -> List[Tuple[str, float]]:
    query_text = add_e5_prefix(question, embedder_key, is_query=True)
    hits = index.similarity_search_with_score(query_text, k=k)
    return [(doc.metadata.get("raw", doc.page_content), float(score)) for doc, score in hits]


sample_chunks = chunk_cache[CONFIGS_TO_RUN[0]["chunker"]][:3]
_ = build_index(sample_chunks, "minilm", "faiss", "check_faiss")
_ = build_index(sample_chunks, "minilm", "chroma", "check_chroma")
print("FAISS and Chroma OK")


## 10. Question set


In [ ]:
import json

BUILTIN_SAMPLE_QUESTIONS = [
    {
        "id": "q1",
        "question": "What problem does synthetic data solve in post-training workflows?",
        "answer": "It scales instruction data by generating examples from teacher models instead of hiring more annotators.",
        "evidence_text": "Instead of hiring more annotators, labs began generating millions of synthetic question-answer pairs, reasoning traces, and tool-use dialogues from stronger teacher models",
    },
    {
        "id": "q2",
        "question": "What role do verifier models play after teacher generation?",
        "answer": "They score or filter teacher outputs and keep only the better candidates.",
        "evidence_text": "A common pattern is generate ten candidate answers, score them with a reranker trained on human preference data or with a lightweight classifier trained to detect unsupported claims, and keep only the top one or two.",
    },
]


def load_questions(path: Optional[str]) -> List[Dict]:
    if path is None or not os.path.exists(path):
        print("No questions file. Using built-in sample (for sample_plain_prose.pdf only).")
        for item in BUILTIN_SAMPLE_QUESTIONS:
            item["ground_truth"] = item["answer"]
        return BUILTIN_SAMPLE_QUESTIONS

    with open(path, encoding="utf-8") as file:
        raw = json.load(file)
    items = raw["questions"] if isinstance(raw, dict) and "questions" in raw else raw

    cleaned = []
    missing = 0
    for item in items:
        item = dict(item)
        if "answer" not in item:
            item["answer"] = item.get("ground_truth", "")
        if "ground_truth" not in item:
            item["ground_truth"] = item.get("answer", "")
        if not collapse_whitespace(str(item.get("evidence_text", ""))):
            missing += 1
        cleaned.append(item)

    if not cleaned:
        raise ValueError("questions file is empty")
    print("Loaded", len(cleaned), "questions from", path)
    if missing:
        print("Warning:", missing, "questions missing evidence_text")
    return cleaned


questions = load_questions(QUESTIONS_PATH)
if SMOKE_TEST:
    questions = questions[:2]
    print("Smoke test: using", len(questions), "questions")

for chunker_name, chunks in chunk_cache.items():
    integrity_by_chunker[chunker_name].update(compute_esi(chunks, questions))

integrity_table = pd.DataFrame(list(integrity_by_chunker.values()))


def format_pct(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "n/a"
    try:
        return f"{float(value):.2f}"
    except (TypeError, ValueError):
        return str(value)


pct_cols = [c for c in integrity_table.columns if c.endswith("_pct")]
display(integrity_table.style.hide(axis="index").format({c: format_pct for c in pct_cols}))


## 11. Answers (extractive by default)


In [ ]:
from typing import Any, List, Optional
import re

import torch
from langchain_core.callbacks.manager import CallbackManagerForLLMRun
from langchain_core.language_models.llms import LLM
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

llm = None


class LocalSeq2SeqLLM(LLM):
    max_new_tokens: int = 128

    def __init__(self, model, tokenizer, device, max_new_tokens: int = 128, **kwargs):
        super().__init__(max_new_tokens=max_new_tokens, **kwargs)
        object.__setattr__(self, "_model", model)
        object.__setattr__(self, "_tokenizer", tokenizer)
        object.__setattr__(self, "_device", device)

    @property
    def _llm_type(self) -> str:
        return "local-seq2seq"

    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> str:
        encoded = self._tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        encoded = {key: value.to(self._device) for key, value in encoded.items()}
        with torch.no_grad():
            generated = self._model.generate(**encoded, max_new_tokens=self.max_new_tokens, do_sample=False)
        text = self._tokenizer.decode(generated[0], skip_special_tokens=True).strip()
        if stop:
            for token in stop:
                if token in text:
                    text = text.split(token)[0]
        return text


def build_local_llm(model_id: str = GENERATOR_MODEL):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
    model.to(device)
    model.eval()
    return LocalSeq2SeqLLM(model, tokenizer, device)


def extractive_answer(question: str, retrieved_texts: List[str]) -> str:
    question_words = set(re.findall(r"[a-z0-9]+", question.lower())) - {
        "what", "which", "why", "how", "does", "do", "the", "a", "an",
        "is", "are", "of", "in", "and", "to", "for", "on", "with",
    }
    best_chunk = retrieved_texts[0]
    best_score = -1
    for chunk in retrieved_texts:
        score = len(question_words & set(re.findall(r"[a-z0-9]+", chunk.lower())))
        if score > best_score:
            best_score = score
            best_chunk = chunk
    sentences = re.split(r"(?<=[.!?])\s+", collapse_whitespace(best_chunk))
    return " ".join(sentences[:2])[:500]


def generate_answer(question: str, retrieved_texts: List[str], llm_model) -> str:
    if not retrieved_texts:
        return "No chunks were retrieved."
    fallback = extractive_answer(question, retrieved_texts)
    if llm_model is None:
        return "[extractive] " + fallback
    context = "\n\n---\n\n".join(retrieved_texts)
    if len(context) > MAX_CONTEXT_CHARS:
        context = context[:MAX_CONTEXT_CHARS] + "\n..."
    prompt = (
        "Question: " + question + "\n"
        "Answer in one or two sentences using only the context.\n\n"
        "Context:\n" + context + "\n\n"
        "Answer:"
    )
    try:
        generated = str(llm_model.invoke(prompt)).strip()
    except Exception as error:
        return "[extractive] " + fallback + " (llm error: " + str(error) + ")"
    if len(generated) < 24:
        return "[extractive] " + fallback
    return generated


if USE_EXTRACTIVE_ONLY:
    print("Extractive answers only (top chunk overlap).")
else:
    print("Loading", GENERATOR_MODEL)
    llm = build_local_llm()


## 12. Scoring helpers

- **Retrieval check:** `evidence_found_in_retrieved` tests substring match for `evidence_text` in top-k chunks.
- **QA proxies:** `compute_local_qa_metrics` fills `context_precision`, `context_recall`, `faithfulness`, `answer_relevancy` using token overlap and MiniLM (`qa_metric_source = local_overlap`).


In [ ]:
import time


def evidence_found_in_retrieved(evidence_text: str, retrieved_texts: List[str]) -> bool:
    evidence = collapse_whitespace(evidence_text)
    if not evidence:
        return False
    return evidence in collapse_whitespace(" ".join(retrieved_texts))


def _word_set(text: str) -> set:
    return set(re.findall(r"[a-z0-9]+", collapse_whitespace(str(text)).lower()))


def _token_recall(reference: str, hypothesis: str) -> float:
    ref = _word_set(reference)
    hyp = _word_set(hypothesis)
    return 0.0 if not ref else len(ref & hyp) / len(ref)


def _token_precision(predicted: str, support: str) -> float:
    pred = _word_set(predicted)
    sup = _word_set(support)
    return 0.0 if not pred else len(pred & sup) / len(pred)


def compute_local_qa_metrics(records: List[Dict]) -> Dict:
    import numpy as np

    if not records:
        return {
            "context_precision": None,
            "context_recall": None,
            "faithfulness": None,
            "answer_relevancy": None,
            "qa_metric_source": "local_overlap",
        }

    embedder = get_embedding_model("minilm")
    question_vectors = np.array(embedder.embed_documents([row["question"] for row in records]))
    answer_vectors = np.array(embedder.embed_documents([row["answer"] for row in records]))

    precisions, recalls, faiths, relevs = [], [], [], []

    for index, row in enumerate(records):
        contexts = row.get("contexts") or []
        combined = " ".join(contexts)
        evidence = collapse_whitespace(str(row.get("evidence_text") or ""))
        gold = evidence if evidence else str(row.get("ground_truth") or "")
        gold_tokens = _word_set(gold)

        if not contexts or not gold_tokens:
            precisions.append(0.0)
        else:
            overlapping = sum(1 for chunk in contexts if gold_tokens & _word_set(chunk))
            precisions.append(overlapping / len(contexts))

        recalls.append(_token_recall(gold, combined))
        faiths.append(_token_precision(row.get("answer") or "", combined))

        qv = question_vectors[index]
        av = answer_vectors[index]
        denom = float(np.linalg.norm(qv)) * float(np.linalg.norm(av)) + 1e-9
        relevs.append(float(np.clip(np.dot(qv, av) / denom, 0.0, 1.0)))

    return {
        "context_precision": round(float(np.mean(precisions)), 4),
        "context_recall": round(float(np.mean(recalls)), 4),
        "faithfulness": round(float(np.mean(faiths)), 4),
        "answer_relevancy": round(float(np.mean(relevs)), 4),
        "qa_metric_source": "local_overlap",
    }


## 13. Run all configurations

Each row is one pipeline. Integrity fields (`sbi_pct`, `mscr_pct`, `esi_pct`) depend only on the chunker; `evidence_hit_rate` is the main comparison metric. See the glossary at the top for all column names.


In [ ]:
def run_one_config(config: Dict, chunks: List[str], question_list: List[Dict]) -> Dict:
    chunker_name = config["chunker"]
    embedder_key = config["embedder"]
    vector_db_name = config["vector_db"]

    print()
    print("=" * 60)
    print("Config:", chunker_name, "+", embedder_key, "+", vector_db_name)
    print("=" * 60)

    start_index_time = time.time()
    collection_name = f"rag_{chunker_name}_{embedder_key}_{vector_db_name}_{int(start_index_time)}"
    index = build_index(chunks, embedder_key, vector_db_name, collection_name)
    index_seconds = time.time() - start_index_time

    records = []
    evidence_hits = 0
    query_seconds_list = []
    retrieved_single_chunk_hits = []

    for item in question_list:
        query_start = time.time()
        hits = retrieve_chunks(index, item["question"], embedder_key, k=TOP_K)
        query_seconds_list.append(time.time() - query_start)

        retrieved_texts = [text for text, score in hits]
        answer = generate_answer(item["question"], retrieved_texts, llm)
        hit = evidence_found_in_retrieved(item.get("evidence_text", ""), retrieved_texts)
        evidence_hits += int(hit)
        retrieved_single_chunk_hits.append(
            int(gold_evidence_fits_in_one_chunk(item.get("evidence_text", ""), retrieved_texts))
        )

        records.append(
            {
                "question": item["question"],
                "answer": answer,
                "contexts": retrieved_texts,
                "ground_truth": item.get("ground_truth") or item.get("answer", ""),
                "evidence_text": item.get("evidence_text", ""),
            }
        )
        print(f"  {item.get('id', '')}: evidence_in_topk={hit} | {answer.replace(chr(10), ' ')[:100]}")

    qa_scores = compute_local_qa_metrics(records)
    integrity = integrity_by_chunker[chunker_name]

    row = {
        "chunker": chunker_name,
        "embedder": embedder_key,
        "vector_db": vector_db_name,
        "num_chunks": integrity["num_chunks"],
        "sbi_pct": integrity["sbi_pct"],
        "mscr_pct": integrity["mscr_pct"],
        "esi_pct": integrity.get("esi_pct"),
        "esi_n": integrity.get("esi_n"),
        "evidence_hit_rate": round(evidence_hits / max(len(question_list), 1), 4),
        "retrieved_esi_rate": round(
            sum(retrieved_single_chunk_hits) / max(len(retrieved_single_chunk_hits), 1), 4
        ),
        "index_time_s": round(index_seconds, 2),
        "avg_query_time_s": round(sum(query_seconds_list) / max(len(query_seconds_list), 1), 4),
        "n_questions": len(question_list),
    }
    row.update(qa_scores)
    return row


def save_row_to_csv(row: Dict, csv_path: str) -> None:
    table = pd.DataFrame([row])
    write_header = not os.path.exists(csv_path)
    table.to_csv(csv_path, mode="a", header=write_header, index=False)


if os.path.exists(RESULTS_CSV):
    os.remove(RESULTS_CSV)

all_rows = []
for config in CONFIGS_TO_RUN:
    result_row = run_one_config(config, chunk_cache[config["chunker"]], questions)
    all_rows.append(result_row)
    save_row_to_csv(result_row, RESULTS_CSV)

results_table = pd.DataFrame(all_rows)

show_cols = [
    "chunker", "embedder", "vector_db", "num_chunks", "sbi_pct", "mscr_pct", "esi_pct",
    "evidence_hit_rate", "retrieved_esi_rate", "index_time_s", "avg_query_time_s",
    "qa_metric_source", "context_precision", "context_recall", "faithfulness", "answer_relevancy",
]
show_cols = [c for c in show_cols if c in results_table.columns]


def format_num(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "n/a"
    try:
        return f"{float(value):.2f}"
    except (TypeError, ValueError):
        return str(value)


float_cols = [c for c in show_cols if results_table[c].dtype.kind in "fc"]
display(results_table[show_cols].style.hide(axis="index").format({c: format_num for c in float_cols}))
print("Saved:", RESULTS_CSV)
try:
    files.download(RESULTS_CSV)
except Exception:
    pass


## 14. Summary tables and correlations

Means use the same column names as `results_table` / `results.csv` (see glossary at the top).


In [ ]:
score_columns = [
    c
    for c in [
        "sbi_pct", "mscr_pct", "esi_pct", "evidence_hit_rate",
        "context_precision", "context_recall", "faithfulness", "answer_relevancy",
    ]
    if c in results_table.columns
]

print("By chunker")
display(results_table.groupby("chunker")[score_columns].mean(numeric_only=True).round(2))

print("By embedder")
display(results_table.groupby("embedder")[score_columns].mean(numeric_only=True).round(2))

print("By vector store")
display(results_table.groupby("vector_db")[score_columns].mean(numeric_only=True).round(2))

if len(results_table) >= 4:
    import numpy as np
    from scipy.stats import spearmanr
    import warnings

    try:
        from scipy.stats import ConstantInputWarning
    except ImportError:
        ConstantInputWarning = UserWarning

    def spearman_pair(column_a, column_b):
        x = pd.to_numeric(results_table[column_a], errors="coerce")
        y = pd.to_numeric(results_table[column_b], errors="coerce")
        valid = x.notna() & y.notna()
        if valid.sum() < 4:
            return "not enough data"
        xv = x[valid].to_numpy(dtype=float)
        yv = y[valid].to_numpy(dtype=float)
        if float(np.nanstd(xv)) == 0.0 or float(np.nanstd(yv)) == 0.0:
            return "undefined (constant column)"
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConstantInputWarning)
            correlation, p_value = spearmanr(xv, yv)
        if correlation != correlation:
            return "undefined"
        return f"rho={correlation:.3f}, p={p_value:.3f}"

    print()
    print("Spearman (exploratory)")
    print("  SBI vs evidence_hit_rate:", spearman_pair("sbi_pct", "evidence_hit_rate"))
    print("  ESI vs evidence_hit_rate:", spearman_pair("esi_pct", "evidence_hit_rate"))


## 15. Figures

Run after section 13 fills `results_table`. Each figure has its own code cell below.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if "results_table" not in globals() or len(results_table) == 0:
    raise RuntimeError("Run section 13 first.")

CHUNKER_ORDER = [c for c in ["fixed", "recursive", "semantic"] if c in results_table["chunker"].unique()]
EMBEDDER_ORDER = [e for e in ["minilm", "bge", "e5"] if e in results_table["embedder"].unique()]
DB_ORDER = [d for d in ["faiss", "chroma"] if d in results_table["vector_db"].unique()]

sns.set_theme(style="whitegrid", font_scale=1.05)
plot_df = results_table.copy()
plot_df["label"] = plot_df["chunker"] + " | " + plot_df["embedder"] + " | " + plot_df["vector_db"]
palette = {"fixed": "#3498db", "recursive": "#9b59b6", "semantic": "#f39c12"}


### Figure 1: Mean evidence hit rate by factor

Three bar charts of mean **`evidence_hit_rate`** (gold `evidence_text` in top-k): grouped by chunker, embedder, and vector store. Dashed line marks the best mean in each panel.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle("Mean evidence hit rate (gold span in top-k)", fontsize=13, fontweight="bold")
panels = [
    ("chunker", CHUNKER_ORDER, "Chunker"),
    ("embedder", EMBEDDER_ORDER, "Embedder"),
    ("vector_db", DB_ORDER, "Vector DB"),
]
colors = sns.color_palette("muted", n_colors=max(len(CHUNKER_ORDER), 3))
for ax, (column, order, title) in zip(axes, panels):
    series = results_table.groupby(column)["evidence_hit_rate"].mean()
    if order:
        series = series.reindex(order)
    bars = ax.bar(range(len(series)), series.values, color=colors[: len(series)], edgecolor="white")
    ax.set_xticks(range(len(series)))
    ax.set_xticklabels(series.index, rotation=15, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("evidence_hit_rate")
    ax.set_title(title)
    ax.axhline(series.max(), color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
    for bar, value in zip(bars, series.values):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 0.02, f"{value:.2f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()


### Figure 2: SBI vs MSCR by chunker

Grouped bars of mean **`sbi_pct`** (Sentence Boundary Integrity) and **`mscr_pct`** (Mid-Sentence Cut Rate) per chunker. These do not vary by embedder or vector DB.


In [ ]:
integrity = (
    results_table.groupby("chunker", as_index=False)[["sbi_pct", "mscr_pct"]]
    .mean()
    .set_index("chunker")
    .reindex(CHUNKER_ORDER)
)
x = range(len(integrity))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar([i - width / 2 for i in x], integrity["sbi_pct"], width, label="sbi_pct (SBI)", color="#2ecc71")
ax.bar([i + width / 2 for i in x], integrity["mscr_pct"], width, label="mscr_pct (MSCR)", color="#e74c3c")
ax.set_xticks(list(x))
ax.set_xticklabels(integrity.index)
ax.set_ylabel("Percent")
ax.set_ylim(0, 110)
ax.set_title("Chunk-boundary integrity by chunker")
ax.legend(loc="upper right")
for i, chunker in enumerate(integrity.index):
    ax.text(i - width / 2, integrity.loc[chunker, "sbi_pct"] + 2, f"{integrity.loc[chunker, 'sbi_pct']:.1f}", ha="center", fontsize=9)
    ax.text(i + width / 2, integrity.loc[chunker, "mscr_pct"] + 2, f"{integrity.loc[chunker, 'mscr_pct']:.1f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()
print("num_chunks by chunker:", results_table.groupby("chunker")["num_chunks"].first().reindex(CHUNKER_ORDER).to_dict())


### Figure 3: Evidence hit rate heatmap (chunker × embedder)

Cell values are mean **`evidence_hit_rate`** averaged over FAISS and Chroma for each chunker/embedder pair.


In [ ]:
heatmap_data = (
    results_table.groupby(["chunker", "embedder"])["evidence_hit_rate"]
    .mean()
    .unstack("embedder")
    .reindex(index=CHUNKER_ORDER, columns=EMBEDDER_ORDER)
)
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"label": "evidence_hit_rate"},
    ax=ax,
)
ax.set_title("evidence_hit_rate: chunker x embedder")
ax.set_xlabel("embedder")
ax.set_ylabel("chunker")
plt.tight_layout()
plt.show()


### Figure 4: SBI vs evidence hit rate (all 18 configs)

Each point is one full pipeline. **x**: `sbi_pct`. **y**: `evidence_hit_rate`. Color: chunker. Tests whether higher boundary integrity tracks retrieval hits.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for chunker in CHUNKER_ORDER:
    subset = plot_df[plot_df["chunker"] == chunker]
    ax.scatter(
        subset["sbi_pct"],
        subset["evidence_hit_rate"],
        s=120,
        alpha=0.85,
        label=chunker,
        color=palette.get(chunker, "gray"),
        edgecolors="white",
        linewidths=0.8,
    )
ax.set_xlabel("sbi_pct (SBI %)")
ax.set_ylabel("evidence_hit_rate")
ax.set_title("All configs: SBI vs evidence hit rate")
ax.set_xlim(-5, 105)
ax.set_ylim(-0.05, 1.05)
ax.legend(title="chunker")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Figure 5: Evidence hit vs local context recall

Horizontal bars per config (`chunker | embedder | vector_db`). Blue: **`evidence_hit_rate`**. Green overlay: **`context_recall`** (local overlap proxy). High recall with low hit rate means related text was retrieved without the exact evidence span.


In [ ]:
if "context_recall" not in results_table.columns:
    print("context_recall not in results_table")
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    plot_df_sorted = plot_df.sort_values("evidence_hit_rate", ascending=True)
    y_pos = range(len(plot_df_sorted))
    ax.barh(y_pos, plot_df_sorted["evidence_hit_rate"], color="#2980b9", alpha=0.85, label="evidence_hit_rate")
    ax.barh(y_pos, plot_df_sorted["context_recall"], color="#27ae60", alpha=0.35, label="context_recall")
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(plot_df_sorted["label"], fontsize=7)
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("Score")
    ax.set_title("Per config: evidence_hit_rate vs context_recall")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
